# Task
Analyze customer support tickets from the `/content/customer_support_tickets.csv` file by cleaning message text, classifying issues into categories like 'PAYMENT', 'LOGIN', 'DELIVERY', 'REFUND', 'BUG', or 'GENERAL' using rule-based logic, and assigning priority levels (P0-P3) based on keywords. Calculate Service Level Agreement (SLA) hours and 'Due Time' for each ticket. Finally, generate a comprehensive report for the support manager summarizing key metrics, export the processed data to `processed_tickets_report.csv`, and summarize the findings with key insights from the analysis.

## Load and Explore Dataset

### Subtask:
Load the customer support tickets dataset from '/content/customer_support_tickets.csv' into a pandas DataFrame. Display the number of rows and columns, identify and count missing values, and show a sample of the tickets to understand the data structure.


**Reasoning**:
To begin, I'll import the pandas library, load the specified CSV file into a DataFrame, print its dimensions, identify and count any missing values, and display the first few rows to get an initial understanding of the data.



In [3]:
import pandas as pd

df = pd.read_csv('/content/customer_support_tickets.csv')

print(f"The DataFrame has {df.shape[0]} rows and {df.shape[1]} columns.\n")

print("Missing values in each column:\n")
print(df.isnull().sum())

print("\nFirst 5 rows of the DataFrame:\n")
print(df.head())

The DataFrame has 8469 rows and 17 columns.

Missing values in each column:

Ticket ID                          0
Customer Name                      0
Customer Email                     0
Customer Age                       0
Customer Gender                    0
Product Purchased                  0
Date of Purchase                   0
Ticket Type                        0
Ticket Subject                     0
Ticket Description                 0
Ticket Status                      0
Resolution                      5700
Ticket Priority                    0
Ticket Channel                     0
First Response Time             2819
Time to Resolution              5700
Customer Satisfaction Rating    5700
dtype: int64

First 5 rows of the DataFrame:

   Ticket ID        Customer Name              Customer Email  Customer Age  \
0          1        Marisa Obrien  carrollallison@example.com            32   
1          2         Jessica Rios    clarkeashley@example.com            42   
2          

## Clean Ticket Messages

### Subtask:
Preprocess the 'Ticket Description' column in the DataFrame by converting text to lowercase, removing special characters, and stripping extra spaces to standardize the text for analysis.


**Reasoning**:
To preprocess the 'Ticket Description' column, I'll convert all text to lowercase, then remove special characters using a regular expression to keep only alphanumeric characters and spaces. Finally, I will strip leading/trailing whitespace and normalize internal spaces to a single space, ensuring the text is standardized for analysis.



In [4]:
df['Ticket Description'] = df['Ticket Description'].str.lower()
df['Ticket Description'] = df['Ticket Description'].str.replace(r'[^a-zA-Z0-9\s]', '', regex=True)
df['Ticket Description'] = df['Ticket Description'].str.strip().str.replace(r'\s+', ' ', regex=True)

print("Preprocessed 'Ticket Description' column (first 5 rows):")
print(df['Ticket Description'].head())

Preprocessed 'Ticket Description' column (first 5 rows):
0    im having an issue with the productpurchased p...
1    im having an issue with the productpurchased p...
2    im facing a problem with my productpurchased t...
3    im having an issue with the productpurchased p...
4    im having an issue with the productpurchased p...
Name: Ticket Description, dtype: object


## Classify Issues

### Subtask:
Implement rule-based logic to classify each ticket into one of the following categories: 'PAYMENT', 'LOGIN', 'DELIVERY', 'REFUND', 'BUG', or 'GENERAL'. This will likely involve searching for keywords within the cleaned ticket messages.


**Reasoning**:
To classify the tickets, I will first define keyword lists for each category, then create a function to apply these rules to the 'Ticket Description' column, and finally display the distribution of the new 'Issue Type' column.



In [5]:
payment_keywords = ['payment', 'charge', 'bill', 'invoice', 'transaction', 'price', 'cost', 'fee', 'money', 'credit', 'debit', 'financial']
login_keywords = ['login', 'account', 'password', 'access', 'username', 'signin', 'authenticate', 'credentials', 'profile']
delivery_keywords = ['delivery', 'shipment', 'order', 'tracking', 'package', 'shipping', 'dispatch', 'receive', 'arrive']
refund_keywords = ['refund', 'return', 'cancel', 'exchange', 'reimburse', 'money back', 'revoke']
bug_keywords = ['bug', 'error', 'glitch', 'crash', 'issue', 'malfunction', 'broken', 'defect', 'problem']

def classify_issue(description):
    if any(keyword in description for keyword in payment_keywords):
        return 'PAYMENT'
    elif any(keyword in description for keyword in login_keywords):
        return 'LOGIN'
    elif any(keyword in description for keyword in delivery_keywords):
        return 'DELIVERY'
    elif any(keyword in description for keyword in refund_keywords):
        return 'REFUND'
    elif any(keyword in description for keyword in bug_keywords):
        return 'BUG'
    else:
        return 'GENERAL'

df['Issue Type'] = df['Ticket Description'].apply(classify_issue)

print("Distribution of 'Issue Type' after classification:")
print(df['Issue Type'].value_counts())

Distribution of 'Issue Type' after classification:
Issue Type
BUG         5909
PAYMENT     1020
LOGIN        740
DELIVERY     607
REFUND       166
GENERAL       27
Name: count, dtype: int64


## Assign Priority Levels

### Subtask:
Based on keywords found in the ticket messages (e.g., 'urgent', 'critical', 'not working'), assign a priority level from P0 (highest) to P3 (lowest) to each ticket. Define the rules for each priority level.


**Reasoning**:
To assign priority levels, I will first define keyword lists for each priority level (P0 to P3), then create a function to apply these rules to the 'Ticket Description' column, and finally apply this function to create a new 'Priority' column and display its distribution.



In [6]:
p0_keywords = ['urgent', 'critical', 'immediate', 'downtime', 'emergency', 'outage']
p1_keywords = ['high priority', 'important', 'not working', 'severe', 'unresponsive', 'broken']
p2_keywords = ['issue', 'problem', 'request', 'slow', 'difficulty', 'error']

def assign_priority(description):
    if any(keyword in description for keyword in p0_keywords):
        return 'P0'
    elif any(keyword in description for keyword in p1_keywords):
        return 'P1'
    elif any(keyword in description for keyword in p2_keywords):
        return 'P2'
    else:
        return 'P3'

df['Priority'] = df['Ticket Description'].apply(assign_priority)

print("Distribution of assigned 'Priority' levels:")
print(df['Priority'].value_counts())

Distribution of assigned 'Priority' levels:
Priority
P2    7866
P1     270
P0     205
P3     128
Name: count, dtype: int64


## Calculate SLA and Due Time

### Subtask:
Calculate the Service Level Agreement (SLA) hours for each ticket based on its assigned priority. Then, determine the 'Due Time' by adding the calculated SLA hours to the ticket creation timestamp. Assume a standard working hour definition if not specified (e.g., 9 AM to 5 PM, weekdays).


**Reasoning**:
To calculate the 'Due Time', I will first convert the 'Date of Purchase' column to datetime objects, as it will serve as the ticket creation timestamp. Then, I'll define a mapping for SLA hours based on priority levels and create a new 'SLA Hours' column by applying this mapping.



In [7]:
df['Date of Purchase'] = pd.to_datetime(df['Date of Purchase'])

sla_mapping = {
    'P0': 4,   # 4 hours
    'P1': 8,   # 8 hours
    'P2': 24,  # 24 hours
    'P3': 48   # 48 hours
}
df['SLA Hours'] = df['Priority'].map(sla_mapping)

print("DataFrame with 'SLA Hours' column (first 5 rows):")
print(df[['Ticket ID', 'Date of Purchase', 'Priority', 'SLA Hours']].head())

DataFrame with 'SLA Hours' column (first 5 rows):
   Ticket ID Date of Purchase Priority  SLA Hours
0          1       2021-03-22       P2         24
1          2       2021-05-22       P2         24
2          3       2020-07-14       P2         24
3          4       2020-11-13       P2         24
4          5       2020-02-04       P2         24


**Reasoning**:
To calculate the 'Due Time' accurately, I will first define the standard working hours and working days. Then, I'll create a function that iteratively adds the SLA hours to the ticket creation time, skipping weekends and non-working hours, to determine the exact 'Due Time'. Finally, I will apply this function to the DataFrame and display the relevant columns to verify the result.



In [8]:
import datetime

# Define working hours (9 AM to 5 PM) and working days (Monday to Friday)
start_hour = 9
end_hour = 17
working_days = [0, 1, 2, 3, 4] # Monday to Friday (0=Monday, 6=Sunday)

def calculate_due_time(created_at, sla_hours):
    current_time = created_at
    remaining_sla = pd.Timedelta(hours=sla_hours)

    while remaining_sla > pd.Timedelta(0):
        # Check if current_time is a working day
        if current_time.weekday() in working_days:
            # Check if current_time is within working hours
            if start_hour <= current_time.hour < end_hour:
                # Calculate time until end of working day or remaining SLA
                end_of_working_day = current_time.replace(hour=end_hour, minute=0, second=0, microsecond=0)
                time_until_eod = end_of_working_day - current_time

                if time_until_eod >= remaining_sla:
                    current_time += remaining_sla
                    remaining_sla = pd.Timedelta(0)
                else:
                    remaining_sla -= time_until_eod
                    current_time = end_of_working_day
            else:
                # If outside working hours, move to the next working hour or day
                if current_time.hour < start_hour: # Before working hours, jump to start_hour
                    current_time = current_time.replace(hour=start_hour, minute=0, second=0, microsecond=0)
                else: # After working hours, jump to next working day's start_hour
                    current_time = (current_time + pd.Timedelta(days=1)).replace(hour=start_hour, minute=0, second=0, microsecond=0)
                    # Skip weekends if they fall on the next day
                    while current_time.weekday() not in working_days:
                        current_time = (current_time + pd.Timedelta(days=1)).replace(hour=start_hour, minute=0, second=0, microsecond=0)
        else:
            # If it's a weekend, jump to the next Monday at start_hour
            current_time = (current_time + pd.Timedelta(days=1)).replace(hour=start_hour, minute=0, second=0, microsecond=0)
            while current_time.weekday() not in working_days:
                current_time = (current_time + pd.Timedelta(days=1)).replace(hour=start_hour, minute=0, second=0, microsecond=0)
    return current_time

# Apply the function to calculate 'Due Time'
df['Due Time'] = df.apply(lambda row: calculate_due_time(row['Date of Purchase'], row['SLA Hours']), axis=1)

print("DataFrame with 'Due Time' column (first 5 rows):")
print(df[['Ticket ID', 'Priority', 'SLA Hours', 'Date of Purchase', 'Due Time']].head())

DataFrame with 'Due Time' column (first 5 rows):
   Ticket ID Priority  SLA Hours Date of Purchase            Due Time
0          1       P2         24       2021-03-22 2021-03-24 17:00:00
1          2       P2         24       2021-05-22 2021-05-26 17:00:00
2          3       P2         24       2020-07-14 2020-07-16 17:00:00
3          4       P2         24       2020-11-13 2020-11-17 17:00:00
4          5       P2         24       2020-02-04 2020-02-06 17:00:00


## Generate Support Manager Report

### Subtask:
Create a comprehensive report summarizing key metrics for the support manager, such as the count of tickets per issue type, average SLA per priority, and any other relevant statistics. Export the final processed DataFrame, including new columns for 'Issue Type', 'Priority', 'SLA Hours', and 'Due Time', to a new CSV file (e.g., 'processed_tickets_report.csv').


**Reasoning**:
To generate the report, I will calculate the number of tickets per issue type and the average SLA hours per priority level, then print these summaries as requested. Finally, I will select the specified columns and export the processed DataFrame to a CSV file.



In [9]:
issue_type_counts = df['Issue Type'].value_counts()
print("\nNumber of tickets per Issue Type:\n")
print(issue_type_counts)

avg_sla_per_priority = df.groupby('Priority')['SLA Hours'].mean()
print("\nAverage SLA Hours per Priority:\n")
print(avg_sla_per_priority)

# Select relevant columns for the report
report_df = df[['Ticket ID', 'Customer Name', 'Ticket Subject', 'Ticket Description', 'Issue Type', 'Priority', 'SLA Hours', 'Due Time', 'Ticket Status', 'Resolution', 'First Response Time', 'Time to Resolution', 'Customer Satisfaction Rating']]

# Export the processed data to a CSV file
report_df.to_csv('processed_tickets_report.csv', index=False)
print("\nProcessed tickets report exported to 'processed_tickets_report.csv'")


Number of tickets per Issue Type:

Issue Type
BUG         5909
PAYMENT     1020
LOGIN        740
DELIVERY     607
REFUND       166
GENERAL       27
Name: count, dtype: int64

Average SLA Hours per Priority:

Priority
P0     4.0
P1     8.0
P2    24.0
P3    48.0
Name: SLA Hours, dtype: float64

Processed tickets report exported to 'processed_tickets_report.csv'


## Final Task

### Subtask:
Summarize the findings from the support ticket analysis, highlighting key insights on issue types, priority distribution, and SLA adherence. Provide the path to the exported CSV report.


## Summary:

### Q&A
*   **Summary of findings on issue types, priority distribution, and SLA adherence:**
    *   **Issue Types**: The majority of tickets are classified as 'BUG' (5909 tickets), followed by 'PAYMENT' (1020 tickets), 'LOGIN' (740 tickets), 'DELIVERY' (607 tickets), 'REFUND' (166 tickets), and a small number as 'GENERAL' (27 tickets).
    *   **Priority Distribution**: Most tickets are assigned 'P2' (7866 tickets), indicating a high volume of moderate priority issues. 'P1' has 270 tickets, 'P0' has 205 tickets, and 'P3' has 128 tickets.
    *   **SLA Adherence**: SLA hours are set based on priority: P0 tickets have a 4-hour SLA, P1 tickets have an 8-hour SLA, P2 tickets have a 24-hour SLA, and P3 tickets have a 48-hour SLA. Due times are calculated considering working hours (9 AM to 5 PM) and weekdays (Monday to Friday).
*   **Path to the exported CSV report:** The processed data has been exported to `processed_tickets_report.csv`.

### Data Analysis Key Findings
*   The dataset contains 8469 customer support tickets with 17 columns.
*   Significant missing values were identified in `Resolution`, `First Response Time`, `Time to Resolution`, and `Customer Satisfaction Rating` (5700 missing values each), and `First Response Time` (2819 missing values).
*   Ticket descriptions were successfully cleaned by converting to lowercase, removing special characters, and standardizing spaces.
*   After classification, 'BUG' is the most frequent issue type, accounting for 5909 tickets.
*   The 'P2' priority level accounts for the vast majority of tickets (7866 out of 8469), indicating a large volume of standard issues.
*   A tiered Service Level Agreement (SLA) was established: P0 tickets with a 4-hour SLA, P1 with 8 hours, P2 with 24 hours, and P3 with 48 hours.
*   'Due Time' for each ticket was accurately calculated by adding the SLA hours to the ticket creation time, factoring in a standard 9 AM - 5 PM workday and excluding weekends.

### Insights or Next Steps
*   The high volume of 'BUG' tickets suggests a potential area for product improvement or enhanced quality assurance to reduce the number of reported defects.
*   The overwhelming number of 'P2' tickets, while having a longer SLA, requires efficient management to prevent backlogs and ensure timely resolution within the 24-hour target.
